# RAG poisoning detection: Colab runner
Run the cells **top to bottom, one at a time**. Before starting: *Runtime → Change runtime type → T4 GPU*.
Durable stuff (small files, caches, results) goes to Google Drive (`MyDrive/ragdet`). The huge raw corpus goes on Colab's
local disk (`/content/data`) and is only needed ONCE, for the *prepare* step. Keep this tab open and in front while cells run.
**Status: the GPU code has never been run before. Expect to hit and fix a few errors on the first tiny run.**

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Connect Google Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ["RAGDET_ROOT"] = "/content/drive/MyDrive/ragdet"        # durable (Drive)
os.environ["RAGDET_DATA"] = "/content/data"                        # big raw corpus (disposable)
os.environ["HF_HOME"]     = "/content/drive/MyDrive/ragdet/hf_cache"
os.environ["PYTHONPATH"]  = "/content/rag-poison-detect/src"
os.makedirs(os.environ["RAGDET_ROOT"], exist_ok=True)
print("paths set")

## 3. Get the code from GitHub (and install what's missing)
Re-run this cell after every disconnect.

In [ ]:
%cd /content
!git clone https://github.com/krishhhna7/rag-poison-detect.git || (cd rag-poison-detect && git pull)
%cd /content/rag-poison-detect
!pip -q install transformers accelerate pyyaml tqdm

## 4. Run the unit tests (should say `23 passed, 1 skipped`)

In [ ]:
!python -m pytest -q

## 5. Download the NQ data (needed only until step 6 has finished once)
Downloads ~1.5 GB to Colab's local disk (a few minutes) and copies the small files to Drive.

In [ ]:
!bash scripts/download_data.sh nq

## 6. Check the files look as expected

In [ ]:
!head -c 400 $RAGDET_DATA/nq/corpus.jsonl
!echo
!head -3 $RAGDET_ROOT/data_small/nq/qrels_test.tsv
!ls -la $RAGDET_ROOT/data_small/nq

## 7. PREPARE (run once): build the small sub-corpus + embed it
Reads the big corpus once (progress bar), keeps ~50,000 passages, embeds them, and saves both to Drive.
After this finishes you never need the 1.5 GB file again. **Takes a while; watch the progress bars.**

In [ ]:
!python -m ragdet.cli prepare --config configs/small_gpu.yaml

### 7b. Confirm it was saved to Drive
You should see a `subcorpus_*.jsonl` (about 30-40 MB) and an `emb_*.npy` (about 150 MB). **Wait one minute after the
previous cell finishes before closing anything**, then re-run this cell to be sure the files are still there.

In [ ]:
!ls -la $RAGDET_ROOT/cache

## 8. TINY run (10 questions)
Finds bugs cheaply and tells us the real speed. Uses its own run name. If a cell errors, copy the **whole** error text.

In [ ]:
!python -m ragdet.cli attack   --config configs/small_gpu.yaml --set data.n_targets=10 run_name=colab_tiny

In [ ]:
!python -m ragdet.cli features --config configs/small_gpu.yaml --set data.n_targets=10 run_name=colab_tiny

In [ ]:
!python -m ragdet.cli a2       --config configs/small_gpu.yaml --set data.n_targets=10 run_name=colab_tiny

In [ ]:
!python -m ragdet.cli evaluate --config configs/small_gpu.yaml --set data.n_targets=10 run_name=colab_tiny

## 9. FULL run (100 questions)
Only after the tiny run worked. After a disconnect: re-run cells 2 and 3, then continue from the stage that did not finish.
If you change attack settings, use a new `run_name` (finished stages are cached).

In [ ]:
!python -m ragdet.cli attack   --config configs/small_gpu.yaml

In [ ]:
!python -m ragdet.cli features --config configs/small_gpu.yaml

In [ ]:
!python -m ragdet.cli a2       --config configs/small_gpu.yaml

In [ ]:
!python -m ragdet.cli evaluate --config configs/small_gpu.yaml

## 10. Look at the results
Files are in Drive: `MyDrive/ragdet/runs/<run_name>/`. Download `results_summary.csv` and `results_raw.csv` and commit them to GitHub.

In [ ]:
import pandas as pd
df = pd.read_csv(os.environ["RAGDET_ROOT"] + "/runs/nq_qwen3b_contriever/results_summary.csv")
df[df.metric.isin(["auroc","tpr","fpr","query_tpr"])]